# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```text
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

# Print publication information
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Version: {getattr(metadata, 'version', 'Unknown')}")
print(f"License: {getattr(metadata, 'license', 'Unknown')}")


## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

**Note:** All references to dataset entities (record sets, fields, columns) use their `@id` values.

In [ ]:
# List all record sets and their @id values

record_sets = dataset.metadata.recordSet
print("Record Sets (@id values and names):")
for rs in record_sets:
    print(f"  @id: {rs['@id']}  name: {rs.get('name', '[Unnamed]')}")

# For each record set, list its fields and columns by @id
for rs in record_sets:
    print(f"\nRecord Set '{rs.get('name', '[Unnamed]')}' (@id={rs['@id']}):")
    # Fields
    fields = rs.get('field', [])
    print("  Fields:")
    for field in fields:
        field_id = field['@id']
        field_name = field.get('name', '[Unnamed]')
        print(f"    @id: {field_id}   name: {field_name}")
        # Columns
        columns = field.get('column', [])
        if columns:
            print("      Columns:")
            for col in columns:
                col_id = col['@id']
                col_name = col.get('name', '[Unnamed]')
                print(f"        @id: {col_id}   name: {col_name}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id` values discovered in the overview step. For demonstration, we'll select the main survey record set and its fields.

In [ ]:
# Collect record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

# Load all records for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")

# For demonstration, show columns of first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns for record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filter records based on specific criteria
- Normalize numeric fields
- Group data by key attributes

We'll use the GAD-7 score (for anxiety), PHQ-9 score (for depression), demographic attributes (such as age and gender), referencing all entities by their `@id`.

In [ ]:
# Example: Use GAD-7 score field (@id) and age group field (@id)
# Replace these with the exact @id values from the overview if needed

# Find likely GAD-7/PHQ-9 score columns and demographic columns
df = dataframes[main_record_set_id]

# Attempt to identify column names corresponding to GAD-7/PHQ-9 scores and age/gender using @id maps
score_cols = [col for col in df.columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower()]
demo_cols = [col for col in df.columns if 'age' in col.lower() or 'gender' in col.lower() or 'education' in col.lower()]

print("Numeric Assessment Score Columns Found:", score_cols)
print("Demographic Columns Found:", demo_cols)

if score_cols:
    # Filter records with GAD-7 score > threshold
    numeric_field = score_cols[0]  # e.g., 'gad7_score'
    threshold = 10
    filtered_df = df[df[numeric_field].astype(float) > threshold]
    print(f"Filtered records where '{numeric_field}' > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
    ) / filtered_df[numeric_field].astype(float).std()

    print(f"Normalized column '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by 'gender' if exists
    group_field = None
    for col in demo_cols:
        if 'gender' in col.lower():
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean '{numeric_field}' by '{group_field}':")
        print(grouped_df.head())


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- Histogram of GAD-7 scores
- Bar plot: Mean GAD-7 scores by gender

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if score_cols:
    numeric_field = score_cols[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].astype(float), bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field}' (GAD-7 score)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Bar plot by gender if present
    group_field = None
    for col in demo_cols:
        if 'gender' in col.lower():
            group_field = col
            break
    if group_field:
        mean_scores = df.groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(6,4))
        sns.barplot(data=mean_scores, x=group_field, y=numeric_field)
        plt.title(f"Mean GAD-7 Score by {group_field}")
        plt.xlabel("Gender")
        plt.ylabel("Mean GAD-7 Score")
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. We reviewed metadata, examined record sets and fields by their `@id` values, extracted tabular data, filtered and normalized assessment scores, and visualized key distributions.

Key insights include demographic and mental health patterns (such as anxiety scores above a threshold, and differences by gender), highlighting the potential for further public health analysis.

All data references were managed using their schema-defined `@id` for reproducibility and standardization.

**Next steps:**
- More complex analyses, such as correlating PHQ-9 and GAD-7 scores.
- Data quality and missingness review.
- Export results for further modeling or reporting.